In [0]:
%sql
USE uk_train_ride.train_rise;

In [0]:
%sql
WITH Date_of_Journey AS (
SELECT
	MONTH(Date_of_Journey) AS MONTH 
	,MONTHNAME(Date_of_Journey) AS MONTHName
	,COUNT(*) AS Current_MONTH
FROM railway
WHERE 
        Journey_Status != 'Cancelled'
GROUP BY 
		MONTH,MONTHName
),
 Laging_difeerence AS (
SELECT 
MONTH
,MONTHName
,Current_MONTH
,LAG(Current_MONTH) OVER(ORDER BY MONTH) As Previous_MOnth 
,ROUND(
(Current_MONTH - LAG(Current_MONTH) OVER(ORDER BY MONTH))/
	LAG(Current_MONTH) OVER(ORDER BY MONTH) * 100,0)AS Growth
FROM Date_of_Journey
)
SELECT 
MONTHName
,Current_MONTH
,Growth
FROM Laging_difeerence
;



In [0]:
%sql
SELECT
 CASE WHEN Railcard = 'None' THEN 'None Card Users' ELSE 'Card Users' END AS Railcards
 ,Count(*) AS Number_of_Users
 ,ROUND(Count(*)/SUM(Count(*)) OVER() * 100,2) AS Percentage
FROM railway
GROUP BY Railcards;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT 
	Railcard 
	,COUNT(*) AS Number_of_Users
  ,ROUND(COUNT(*)/SUM(COUNT(*)) OVER()* 100,2) as Percentage
FROM railway
WHERE Railcard != 'None'
GROUP BY Railcard;

In [0]:
%sql
SELECT 
	Ticket_Class
    ,COUNT(*) AS Number_of_CardHolders
    ,ROUND(Count(*)/SUM(Count(*)) OVER() * 100,2) AS Percentage
FROM railway
GROUP BY Ticket_Class;



In [0]:
%sql
SELECT 
    Ticket_Type
    ,COUNT(*) AS Number_of_CardHolder
    ,ROUND(Count(*)/SUM(Count(*)) OVER() * 100,2) AS Percentage
FROM railway
GROUP BY 
      Ticket_Type
ORDER BY 
      Number_of_CardHolder
      ,Percentage;

In [0]:
%sql
SELECT 
    CONCAT(CAST(SPLIT(Arrival_Time, ':')[0] AS INT), ':00 AM') AS Hour,
    COUNT(*) AS Number_of_Journeys
FROM railway
WHERE Arrival_Time LIKE '%AM%' 
AND  
Journey_Status != 'Cancelled'
GROUP BY CAST(SPLIT(Arrival_Time, ':')[0] AS INT)
ORDER BY CAST(SPLIT(Arrival_Time, ':')[0] AS INT);

In [0]:
%sql
SELECT 
      DAY(TRY_CAST(Date_of_Journey AS DATE)) AS DAYNUM
      ,DAYNAME(TRY_CAST(Date_of_Journey AS DATE)) AS Day
      ,CAST(split(Departure_Time,':')[0] As INT) as Hour_num
      ,CONCAT(CAST(split(Departure_Time,':')[0] As INT), ':00 AM') AS Hour
      ,COUNT(*) AS COUNT
FROM railway
WHERE
      Journey_Status != 'Cancelled'
GROUP BY 
      DAY(TRY_CAST(Date_of_Journey AS DATE))
      ,DAYNAME(TRY_CAST(Date_of_Journey AS DATE))
      ,CAST(split(Departure_Time,':')[0] As INT)
      ,CONCAT(CAST(split(Departure_Time,':')[0] As INT), ':00 AM');

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
WITH Monthly_Journey_Count_by_Hour_AM AS (
SELECT 
    Month(Date_of_Journey)
    ,MONTHNAME(Date_of_Journey) AS Month
    ,CAST(split(Departure_Time,':') [0] AS INT) AS HOUR_NUM
    ,CONCAT(CAST(split(Departure_Time,':') [0] AS INT),':00 AM') AS Hour
    ,COUNT(*) AS Number_of_Passengers
FROM railway
WHERE 
        Departure_Time LIKE '%AM%' 
        AND  
        Journey_Status != 'Cancelled'
GROUP BY 
    Month(Date_of_Journey)
    , MONTHNAME(Date_of_Journey)
    ,HOUR_NUM,Hour
ORDER BY 
Month(Date_of_Journey)
,CAST(split(Departure_Time,':') [0] AS INT)
)
    SELECT 
    Month
    ,Hour 
    ,Number_of_Passengers
FROM Monthly_Journey_Count_by_Hour_AM;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
WITH Monthly_Journey_Count_by_Hour_AM AS (
SELECT 
    Month(Date_of_Journey)
    ,MONTHNAME(Date_of_Journey) AS Month
    ,CAST(split(Departure_Time,':') [0] AS INT) AS HOUR_NUM
    ,CONCAT(CAST(split(Departure_Time,':') [0] AS INT),':00 PM') AS Hour
    ,COUNT(*) AS Number_of_Passengers
FROM railway
WHERE 
        Departure_Time LIKE '%PM%'
        AND  
        Journey_Status != 'Cancelled'
GROUP BY 
    Month(Date_of_Journey)
    , MONTHNAME(Date_of_Journey)
    ,HOUR_NUM,Hour
ORDER BY 
Month(Date_of_Journey)
,CAST(split(Departure_Time,':') [0] AS INT)
)
    SELECT 
    Month
    ,Hour 
    ,Number_of_Passengers
FROM Monthly_Journey_Count_by_Hour_AM;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql

WITH Station_Hourly AS (
    SELECT 
        Departure_Station,
        CAST(SPLIT(Departure_Time, ':')[0] AS INT) AS Time_Num,
        CONCAT(
            CAST(SPLIT(Departure_Time, ':')[0] AS INT),
            ':00 AM'
        ) AS Time,
        COUNT(*) AS Number_Of_Passengers
    FROM railway
    WHERE 
            Departure_Time LIKE '%AM' AND  
        Journey_Status != 'Cancelled'
    GROUP BY 
        Departure_Station,
        SPLIT(Departure_Time, ':')[0]
),

Station_Total AS (
    SELECT
        Departure_Station,
        SUM(Number_Of_Passengers) AS Total_Passengers
    FROM Station_Hourly
    GROUP BY Departure_Station
),

Top_7_Stations AS (
    SELECT
        Departure_Station,
        Total_Passengers,
        ROW_NUMBER() OVER (
            ORDER BY Total_Passengers DESC
        ) AS RNK
    FROM Station_Total
)

SELECT
    h.Departure_Station,
    h.Time_Num,
    h.Time,
    h.Number_Of_Passengers,
    t.Total_Passengers
FROM Station_Hourly h
JOIN Top_7_Stations t
    ON h.Departure_Station = t.Departure_Station
WHERE t.RNK <= 7
ORDER BY
    t.Total_Passengers DESC,
    h.Time_Num;

In [0]:
%sql

WITH Station_Hourly AS (
    SELECT 
        Departure_Station,
        CAST(SPLIT(Departure_Time, ':')[0] AS INT) AS Time_Num,
        CONCAT(
            CAST(SPLIT(Departure_Time, ':')[0] AS INT),
            ':00 PM'
        ) AS Time,
        COUNT(*) AS Number_Of_Passengers
    FROM railway
    WHERE 
            Departure_Time LIKE '%PM' AND  
        Journey_Status != 'Cancelled'
    GROUP BY 
        Departure_Station,
        SPLIT(Departure_Time, ':')[0]
),

Station_Total AS (
    SELECT
        Departure_Station,
        SUM(Number_Of_Passengers) AS Total_Passengers
    FROM Station_Hourly
    GROUP BY Departure_Station
),

Top_7_Stations AS (
    SELECT
        Departure_Station,
        Total_Passengers,
        ROW_NUMBER() OVER (
            ORDER BY Total_Passengers DESC
        ) AS RNK
    FROM Station_Total
)

SELECT
    h.Departure_Station,
    h.Time_Num,
    h.Time,
    h.Number_Of_Passengers,
    t.Total_Passengers
FROM Station_Hourly h
JOIN Top_7_Stations t
    ON h.Departure_Station = t.Departure_Station
WHERE t.RNK <= 7
ORDER BY
    t.Total_Passengers DESC,
    h.Time_Num;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Top 5 Stations Comparison: AM vs PM
WITH AM_Stations AS (
    SELECT 
        Departure_Station,
        COUNT(*) AS AM_Passengers
    FROM railway
    WHERE 
            Departure_Time LIKE '%AM'
            AND  
        Journey_Status != 'Cancelled'
    GROUP BY Departure_Station
    ORDER BY AM_Passengers DESC
    LIMIT 5
),
PM_Stations AS (
    SELECT 
        Departure_Station,
        COUNT(*) AS PM_Passengers
    FROM railway
    WHERE 
            Departure_Time LIKE '%PM'
            AND  
            Journey_Status != 'Cancelled'
    GROUP BY Departure_Station
    ORDER BY PM_Passengers DESC
    LIMIT 5
)
SELECT 
    COALESCE(a.Departure_Station, p.Departure_Station) AS Station,
    COALESCE(a.AM_Passengers, 0) AS AM_Passengers,
    COALESCE(p.PM_Passengers, 0) AS PM_Passengers
FROM AM_Stations a
FULL OUTER JOIN PM_Stations p ON a.Departure_Station = p.Departure_Station
ORDER BY (COALESCE(a.AM_Passengers, 0) + COALESCE(p.PM_Passengers, 0)) DESC

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

# Define consistent color scheme
COLORS = {
    'primary': '#1f77b4',
    'secondary': '#ff7f0e', 
    'tertiary': '#2ca02c',
    'quaternary': '#d62728',
    'purple': '#9467bd',
    'brown': '#8c564b'
}

# Get data from Cell 2
df_trend = spark.sql("""
WITH Date_of_Journey AS (
    SELECT
        MONTH(Date_of_Journey) AS MONTH,
        MONTHNAME(Date_of_Journey) AS MONTHName,
        COUNT(*) AS Current_MONTH
    FROM uk_train_ride.train_rise.railway
    WHERE Journey_Status != 'Cancelled'
    GROUP BY MONTH, MONTHName
),
Laging_difeerence AS (
    SELECT 
        MONTH,
        MONTHName,
        Current_MONTH,
        LAG(Current_MONTH) OVER(ORDER BY MONTH) As Previous_MOnth,
        ROUND((Current_MONTH - LAG(Current_MONTH) OVER(ORDER BY MONTH))/
            LAG(Current_MONTH) OVER(ORDER BY MONTH) * 100, 0) AS Growth
    FROM Date_of_Journey
)
SELECT MONTHName, Current_MONTH, Growth
FROM Laging_difeerence
""").toPandas()

# Create figure with subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Monthly Passenger Journey Analysis', fontsize=16, fontweight='bold')

# Plot 1: Journey Count Trend
ax1.plot(df_trend['MONTHName'], df_trend['Current_MONTH'], 
         marker='o', linewidth=2.5, markersize=10, color=COLORS['primary'])
ax1.fill_between(range(len(df_trend)), df_trend['Current_MONTH'], alpha=0.3, color=COLORS['primary'])
ax1.set_title('Monthly Journey Count', fontsize=14, fontweight='bold')
ax1.set_xlabel('Month', fontsize=12)
ax1.set_ylabel('Number of Journeys', fontsize=12)
ax1.grid(True, alpha=0.3)
for i, v in enumerate(df_trend['Current_MONTH']):
    ax1.text(i, v + 100, f'{v:,}', ha='center', va='bottom', fontsize=10)

# Plot 2: Growth Rate
colors = [COLORS['tertiary'] if x > 0 else COLORS['quaternary'] if x < 0 else 'gray' 
          for x in df_trend['Growth'].fillna(0)]
ax2.bar(df_trend['MONTHName'], df_trend['Growth'].fillna(0), color=colors, alpha=0.8)
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
ax2.set_title('Month-over-Month Growth Rate (%)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Month', fontsize=12)
ax2.set_ylabel('Growth (%)', fontsize=12)
ax2.grid(True, alpha=0.3, axis='y')
for i, v in enumerate(df_trend['Growth'].fillna(0)):
    if v != 0:
        ax2.text(i, v + (1 if v > 0 else -1), f'{v:.0f}%', ha='center', 
                va='bottom' if v > 0 else 'top', fontsize=10)

plt.tight_layout()
display(plt.show())

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

# Get railcard data
df_railcard = spark.sql("""
SELECT
    CASE WHEN Railcard = 'None' THEN 'None Card Users' ELSE 'Card Users' END AS Railcards,
    Count(*) AS Number_of_Users,
    ROUND(Count(*)/SUM(Count(*)) OVER() * 100, 2) AS Percentage
FROM uk_train_ride.train_rise.railway
GROUP BY Railcards
""").toPandas()

df_railcard_types = spark.sql("""
SELECT 
    Railcard,
    COUNT(*) AS Number_of_Users,
    ROUND(COUNT(*)/SUM(COUNT(*)) OVER()* 100, 2) as Percentage
FROM uk_train_ride.train_rise.railway
WHERE Railcard != 'None'
GROUP BY Railcard
""").toPandas()

# Create figure with subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Railcard Usage Analysis', fontsize=16, fontweight='bold')

# Plot 1: Overall Railcard vs Non-Railcard
colors1 = [COLORS['primary'], COLORS['secondary']]
wedges, texts, autotexts = ax1.pie(df_railcard['Number_of_Users'], 
                                     labels=df_railcard['Railcards'],
                                     autopct='%1.1f%%',
                                     colors=colors1,
                                     startangle=90,
                                     textprops={'fontsize': 12})
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
ax1.set_title('Railcard vs Non-Railcard Users', fontsize=14, fontweight='bold')

# Plot 2: Railcard Type Breakdown
colors2 = [COLORS['primary'], COLORS['tertiary'], COLORS['purple']]
bars = ax2.bar(df_railcard_types['Railcard'], df_railcard_types['Number_of_Users'],
               color=colors2, alpha=0.8)
ax2.set_title('Railcard Type Distribution', fontsize=14, fontweight='bold')
ax2.set_xlabel('Railcard Type', fontsize=12)
ax2.set_ylabel('Number of Users', fontsize=12)
ax2.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height):,}\n({df_railcard_types.loc[bars.index(bar), "Percentage"]:.1f}%)',
            ha='center', va='bottom', fontsize=10)

plt.tight_layout()
display(plt.show())

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

# Get AM/PM station comparison data
df_comparison = spark.sql("""
WITH AM_Stations AS (
    SELECT 
        Departure_Station,
        COUNT(*) AS AM_Passengers
    FROM uk_train_ride.train_rise.railway
    WHERE Departure_Time LIKE '%AM'
        AND Journey_Status != 'Cancelled'
    GROUP BY Departure_Station
    ORDER BY AM_Passengers DESC
    LIMIT 5
),
PM_Stations AS (
    SELECT 
        Departure_Station,
        COUNT(*) AS PM_Passengers
    FROM uk_train_ride.train_rise.railway
    WHERE Departure_Time LIKE '%PM'
        AND Journey_Status != 'Cancelled'
    GROUP BY Departure_Station
    ORDER BY PM_Passengers DESC
    LIMIT 5
)
SELECT 
    COALESCE(a.Departure_Station, p.Departure_Station) AS Station,
    COALESCE(a.AM_Passengers, 0) AS AM_Passengers,
    COALESCE(p.PM_Passengers, 0) AS PM_Passengers
FROM AM_Stations a
FULL OUTER JOIN PM_Stations p ON a.Departure_Station = p.Departure_Station
ORDER BY (COALESCE(a.AM_Passengers, 0) + COALESCE(p.PM_Passengers, 0)) DESC
""").toPandas()

# Create grouped bar chart
fig, ax = plt.subplots(figsize=(14, 6))
fig.suptitle('Top 5 Stations: AM vs PM Peak Hours', fontsize=16, fontweight='bold')

x = range(len(df_comparison))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], df_comparison['AM_Passengers'], 
               width, label='AM Peak', color=COLORS['primary'], alpha=0.8)
bars2 = ax.bar([i + width/2 for i in x], df_comparison['PM_Passengers'], 
               width, label='PM Off-Peak', color=COLORS['secondary'], alpha=0.8)

ax.set_xlabel('Departure Station', fontsize=12)
ax.set_ylabel('Number of Passengers', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(df_comparison['Station'], rotation=45, ha='right')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{int(height):,}',
                   ha='center', va='bottom', fontsize=9)

plt.tight_layout()
display(plt.show())